In [1]:
import geopandas as gpd
import pandas as pd
import osmnx as ox
import networkx as nx
import matplotlib.pyplot as plt
from shapely.geometry import LineString, Point, Polygon, MultiPolygon

ox.__version__

'2.0.3'

In [9]:
network_type = 'walk'
# custom filter for building walk network
cf = """
     ["area"!~"yes"]
     ["highway"]
     ["highway"!~"motor|proposed|construction|abandoned|platform|raceway"]
     ["foot"!~"no"]
     ["service"!~"private"]
     ["access"!~"private"]
     """
trip_times = [5, 10, 15]  # in minutes
travel_speed = 4.5  # walking speed in km/hour
# define a bounding box in Blacktown Suburbs as (left, bottom, right, top)
bbox = 150.868721,-33.766445,150.974464,-33.692731

In [10]:
# create network from that bounding box
G = ox.graph_from_bbox(bbox, custom_filter=cf, network_type=network_type)

In [ ]:
# explore nodes and edges together in a single map
nodes, edges = ox.graph_to_gdfs(G)
m = edges.explore(color="skyblue", tiles="cartodbdarkmatter")
nodes.explore(m=m, color="pink", marker_kwds={"radius": 6})



## Network Setup for Walksheds

In [11]:
# project the graph to UTM
G = ox.project_graph(G)

# add an edge attribute for time in minutes required to traverse each edge
meters_per_minute = travel_speed * 1000 / 60  # km per hour to m per minute
for _, _, _, data in G.edges(data=True, keys=True):
    data["time"] = data["length"] / meters_per_minute

# get one color for each isochrone
iso_colors = ox.plot.get_colors(n=len(trip_times), cmap="plasma", start=0)


In [12]:
# ------ Input points for walksheds ------
suburb_stops_df = pd.read_csv("data/suburb_stops.csv")

# convert stops to a geodataframe
gdf_stops = gpd.GeoDataFrame(suburb_stops_df, geometry=gpd.points_from_xy(suburb_stops_df.lon, suburb_stops_df.lat), crs="EPSG:4326")

# drop lat and lon from properties
gdf_stops = gdf_stops.drop(columns=['lat', 'lon'])

# Export to GeoJSON (before reprojecting to graph CRS)
gdf_stops.to_file("walksheds/bus_stops.geojson", driver="GeoJSON")

# Reproject stops to match graph CRS
gdf_stops = gdf_stops.to_crs(G.graph['crs'])

In [20]:
# helper function for creating walksheds
def make_iso_polys(G, origin, trip_times, stop_id, suburb_name, edge_buff=25, node_buff=50, infill=False):
    isochrone_polys = []

    # normalize trip_times to always be a list
    if isinstance(trip_times, (int, float)):
        trip_times = [trip_times]
    
    for trip_time in trip_times:
        subgraph = nx.ego_graph(G, origin, radius=trip_time, distance="time")

        node_points = [Point((data["x"], data["y"])) for node, data in subgraph.nodes(data=True)]
        nodes_gdf = gpd.GeoDataFrame({"id": list(subgraph.nodes)}, geometry=node_points)
        nodes_gdf = nodes_gdf.set_index("id")

        edge_lines = []
        for n_fr, n_to in subgraph.edges():
            f = nodes_gdf.loc[n_fr].geometry
            t = nodes_gdf.loc[n_to].geometry

            edge_data = G.get_edge_data(n_fr, n_to)
            if edge_data:
                # Safely get the first edge’s geometry, or fallback to straight line
                first_edge = list(edge_data.values())[0]
                edge_geom = first_edge.get("geometry", LineString([f, t]))
            else:
                edge_geom = LineString([f, t])

            edge_lines.append(edge_geom)
            
            # edge_lookup = G.get_edge_data(n_fr, n_to)[0].get("geometry", LineString([f, t]))
            # edge_lines.append(edge_lookup)

        n = nodes_gdf.buffer(node_buff).geometry
        e = gpd.GeoSeries(edge_lines).buffer(edge_buff).geometry
        all_gs = list(n) + list(e)
        new_iso = gpd.GeoSeries(all_gs).union_all()

        if infill:
            if isinstance(new_iso, (MultiPolygon, Polygon)):
                new_iso = Polygon(new_iso.exterior)

        isochrone_polys.append(
        {
            'stop_id': stop_id,
            'suburb': suburb_name,
            'walk_time': trip_time,
            'geometry': new_iso
        })

        # convert to a pandas df
        iso_df = pd.DataFrame(isochrone_polys, columns=['stop_id', 'suburb', 'walk_time', 'geometry'])

    return iso_df

In [6]:
def add_point_on_edge_as_node(G, lon, lat):
    """
    Adds a point on the nearest edge of the graph as a new node,
    splits the edge into two, and returns the updated graph and new node ID.
    
    Parameters:
        G (networkx.MultiDiGraph): The street network graph.
        point_latlon (tuple): The origin point in (lat, lon) format.
    
    Returns:
        G (networkx.MultiDiGraph): Modified graph with new node.
        new_node (int): ID of the new node added on the edge.
    """
    G = G.copy()
    
    # Find nearest edge
    u, v, key = ox.distance.nearest_edges(G, lon, lat)
    edge_data = G[u][v][key]
    
    # Get geometry of edge
    if 'geometry' in edge_data:
        edge_geom = edge_data['geometry']
    else:
        # If no geometry, create straight line between nodes
        point_u = Point((G.nodes[u]['x'], G.nodes[u]['y']))
        point_v = Point((G.nodes[v]['x'], G.nodes[v]['y']))
        edge_geom = LineString([point_u, point_v])
    
    # Interpolate point along edge
    point = Point(lon, lat)
    new_point = edge_geom.interpolate(edge_geom.project(point))
    
    # Add new node
    new_node_id = max(G.nodes) + 1
    G.add_node(new_node_id, x=new_point.x, y=new_point.y)
    
    # Remove old edge
    G.remove_edge(u, v, key)
    
    # Add new edges with adjusted lengths
    geom1 = LineString([Point((G.nodes[u]['x'], G.nodes[u]['y'])), new_point])
    geom2 = LineString([new_point, Point((G.nodes[v]['x'], G.nodes[v]['y']))])
    
    attrs1 = edge_data.copy()
    attrs2 = edge_data.copy()
    attrs1['geometry'] = geom1
    attrs2['geometry'] = geom2
    attrs1['length'] = geom1.length
    attrs2['length'] = geom2.length

    G.add_edge(u, new_node_id, **attrs1)
    G.add_edge(new_node_id, v, **attrs2)

    return G, new_node_id

In [ ]:
print("creating walkshed polygons...")

# Process all stops
features = []

# for each stop, generate a node on the closest edge, then pass that network into the make isopoly function
for idx, row in gdf_stops.iterrows():
    center_node = ox.distance.nearest_nodes(G, row.geometry.x, row.geometry.y)
    iso_polys = make_iso_polys(G, center_node, trip_times, row.stop_id, row.suburb_name, edge_buff=25, node_buff=0, infill=True)

    features.append(iso_polys)
    

print("done.")

# use concat with ignore index to remove duplicate column names and index from 0 to n-1
final_df = pd.concat(features, ignore_index=True)

# Create GeoDataFrame
iso_gdf = gpd.GeoDataFrame(final_df, crs=G.graph['crs'])

# Reproject to EPSG:4326 (WGS84 lat/lon)
iso_gdf = iso_gdf.to_crs("EPSG:4326")

print("outputting GeoJSON to 'bus_stop_isochrones.geojson'")

# Export to GeoJSON
iso_gdf.to_file("walksheds/bus_stop_isochrones.geojson", driver="GeoJSON")

creating walkshed polygons...
done.
outputting GeoJSON to 'bus_stop_isochrones.geojson'


## Create 400 meter buffers

In [38]:
time_to_walk_400m = 5.33333



select_stops = gdf_stops[gdf_stops['stop_id'] == 276369]

print(select_stops)

# Process all stops
features = []
new_nodes = []

for idx, row in select_stops.iterrows():
    G_updated, origin = add_point_on_edge_as_node(G, row.geometry.x, row.geometry.y)

    center_node = ox.distance.nearest_nodes(G_updated, row.geometry.x, row.geometry.y)

    x = G_updated.nodes[center_node]['x']  # longitude
    y = G_updated.nodes[center_node]['y']  # latitude

    temp_node = {
        'stop_id': row.stop_id,
        'lon': x,
        'lat': y
    }

    # convert to a pandas df
    new_nodes.append(temp_node)

    if(row.stop_id == 276369):
        nodes, edges = ox.graph_to_gdfs(G_updated)

        print(nodes)
    

    # iso_polys = make_iso_polys(G_updated, origin, time_to_walk_400m, row.stop_id, row.suburb_name, edge_buff=25, node_buff=0, infill=True)

    # features.append(iso_polys)

new_nodes = pd.DataFrame(new_nodes, columns=['stop_id', 'lon', 'lat'])

# Create GeoDataFrame
new_nodes_df = gpd.GeoDataFrame(new_nodes, geometry=gpd.points_from_xy(new_nodes.lon, new_nodes.lat), crs=G.graph['crs'])

# drop lat and lon columns
new_nodes_df = new_nodes_df.drop(columns=['lon', 'lat'])

# Reproject to EPSG:4326 (WGS84 lat/lon)
new_nodes_df = new_nodes_df.to_crs("EPSG:4326")

# Export to GeoJSON
new_nodes_df.to_file("walksheds/stop_nodes.geojson", driver="GeoJSON")

"""
# use concat with ignore index to remove duplicate column names and index from 0 to n-1
final_df = pd.concat(features, ignore_index=True)

# Create GeoDataFrame
iso_gdf = gpd.GeoDataFrame(final_df, crs=G.graph['crs'])

# Reproject to EPSG:4326 (WGS84 lat/lon)
iso_gdf = iso_gdf.to_crs("EPSG:4326")

print("outputting GeoJSON to 'walkshed_400m.geojson'")

# Export to GeoJSON
iso_gdf.to_file("walksheds/walkshed_400m_new.geojson", driver="GeoJSON")
"""

   id  stop_id     suburb_name                        geometry
5   6   276369  Acacia Gardens  POINT (306279.696 6265787.656)
                        y              x  street_count   highway  ref railway  \
osmid                                                                           
13148560     6.262088e+06  306729.729446           4.0       NaN  NaN     NaN   
76468428     6.262084e+06  306745.788075           4.0       NaN  NaN     NaN   
11292973579  6.262103e+06  306733.851487           4.0  crossing  NaN     NaN   
11292973573  6.262072e+06  306725.907310           4.0  crossing  NaN     NaN   
11292973570  6.262091e+06  306721.036243           4.0  crossing  NaN     NaN   
...                   ...            ...           ...       ...  ...     ...   
12829481925  6.263363e+06  307111.918704           3.0       NaN  NaN     NaN   
12825873986  6.262729e+06  307293.069824           1.0       NaN  NaN     NaN   
12825874004  6.262690e+06  307481.474644           1.0       NaN

'\n# use concat with ignore index to remove duplicate column names and index from 0 to n-1\nfinal_df = pd.concat(features, ignore_index=True)\n\n# Create GeoDataFrame\niso_gdf = gpd.GeoDataFrame(final_df, crs=G.graph[\'crs\'])\n\n# Reproject to EPSG:4326 (WGS84 lat/lon)\niso_gdf = iso_gdf.to_crs("EPSG:4326")\n\nprint("outputting GeoJSON to \'walkshed_400m.geojson\'")\n\n# Export to GeoJSON\niso_gdf.to_file("walksheds/walkshed_400m_new.geojson", driver="GeoJSON")\n'

In [27]:
# Import bus stop inventory csv
stop_inventory_df = pd.read_csv("data/suburb_bus_stop_inventory.csv")

# calculate number of amenities and create new field with this information
for indx, val in stop_inventory_df.iterrows():
    stop_inventory_df['num_amenities'] = stop_inventory_df.iloc[:, :].eq('Yes').sum(axis=1)

# join to suburb_stops dataframe to get lat/lon for each stop
stop_inventory_df = pd.merge(stop_inventory_df, suburb_stops_df[['stop_id', 'lat', 'lon']], on='stop_id', how='left')

# convert to geodataframe
gdf_stop_inventory = gpd.GeoDataFrame(stop_inventory_df, geometry=gpd.points_from_xy(stop_inventory_df.lon, stop_inventory_df.lat), crs="EPSG:4326")

# drop lat and lon from properties
gdf_stop_inventory = gdf_stop_inventory.drop(columns=['lat', 'lon'])

# export to GeoJSON
gdf_stop_inventory.to_file("walksheds/bus_stop_inventory.geojson", driver="GeoJSON")